In [1]:
import sys
sys.path.append("../..")

from pathlib import Path

import pandas as pd
from truckscenes import TruckScenes

from src.config import DATA_ROOT, METADATA_ROOT, SENSOR_ROOT
from src.dataset_filtering import prepare_available_sample_data
from src.sensor_observation import build_sensor_data_df
from src.annotation_velocity import build_vehicle_reference_df
from src.radar_velocity import (
    build_target_radar_points_df,
    build_radar_radial_df,
    build_radar_velocity_df,
    build_radar_radial_validation_df,
)

RESULT_DIR = Path("../../results/condition")

In [2]:
trucksc = TruckScenes(dataroot=DATA_ROOT)

available_sample_data, summary, extra_files = prepare_available_sample_data()
display(summary)

sensor_data_df = build_sensor_data_df(
    metadata_root=METADATA_ROOT,
    sample_data_df=available_sample_data,
)

keyframe_df = sensor_data_df[
    sensor_data_df["Is Key Frame"].eq(True)
].copy()

radar_observation_df = (
    keyframe_df[keyframe_df["Modality"].eq("RADAR")]
    .reset_index(drop=True)
)

print("RADAR observations:", len(radar_observation_df))

Auto-detected latest version: v1.2-trainval
Loading truckscenes tables for version v1.2-trainval...
11 attribute,
18 calibrated_sensor,
27 category,
1200243 ego_motion_cabin,
1200183 ego_motion_chassis,
1201993 ego_pose,
43769 instance,
23902 sample,
819536 sample_annotation,
2586264 sample_data,
598 scene,
18 sensor,
4 visibility,
598 weather_annotation,
Done loading in 48.329 seconds.
Reverse indexing ...
Done reverse indexing in 20.7 seconds.
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Metadata records                    2586264
Available local files                820213
Unavailable locally                 1766051
Extra / unreferenced local files          2
Name: Dataset Filtering Summary, dtype: int64

RADAR observations: 60378


In [3]:
vehicle_reference_df = build_vehicle_reference_df(
    trucksc.sample,
    trucksc.sample_annotation,
    trucksc.instance,
    trucksc.category,
)

print("Reference intervals:", len(vehicle_reference_df))

Reference intervals: 534039


In [4]:
vehicle_reference_df.to_csv(
    RESULT_DIR / "vehicle_reference_df.csv",
    index=False,
)

In [5]:
radar_tokens = set(radar_observation_df["Token"])
radar_sample_tokens = set(radar_observation_df["Sample Token"])

radar_sample_data = available_sample_data[
    available_sample_data["token"].isin(radar_tokens)
].copy()

radar_sample_data_lookup = (
    radar_sample_data.set_index("token").to_dict("index")
)

ego_pose_tokens = set(radar_sample_data["ego_pose_token"].dropna())
radar_ego_pose_lookup = {
    x["token"]: x
    for x in trucksc.ego_pose
    if x["token"] in ego_pose_tokens
}

calibrated_sensor_tokens = set(
    radar_sample_data["calibrated_sensor_token"].dropna()
)
radar_calibrated_sensor_lookup = {
    x["token"]: x
    for x in trucksc.calibrated_sensor
    if x["token"] in calibrated_sensor_tokens
}

radar_annotations = [
    x
    for x in trucksc.sample_annotation
    if x["sample_token"] in radar_sample_tokens
]

In [6]:
target_radar_points_df = build_target_radar_points_df(
    radar_observation_df,
    radar_annotations,
    trucksc.instance,
    trucksc.category,
    radar_sample_data_lookup,
    radar_ego_pose_lookup,
    radar_calibrated_sensor_lookup,
    SENSOR_ROOT,
)

print("Target RADAR points:", len(target_radar_points_df))

Target RADAR points: 759847


In [7]:
radar_radial_df = build_radar_radial_df(
    target_radar_points_df,
    radar_sample_data_lookup,
    radar_ego_pose_lookup,
    radar_calibrated_sensor_lookup,
    trucksc.ego_motion_cabin,
)

print("RADAR radial points:", len(radar_radial_df))

RADAR radial points: 759847


In [8]:
target_radar_points_df = build_target_radar_points_df(
    radar_observation_df,
    radar_annotations,
    trucksc.instance,
    trucksc.category,
    radar_sample_data_lookup,
    radar_ego_pose_lookup,
    radar_calibrated_sensor_lookup,
    SENSOR_ROOT,
)

In [9]:
radar_radial_df.to_csv(
    RESULT_DIR / "radar_radial_df.csv",
    index=False,
)

In [10]:
radar_velocity_df = build_radar_velocity_df(
    radar_radial_df,
    vehicle_reference_df,
)

print("RADAR reconstructable intervals:", len(radar_velocity_df))
print("RADAR stable intervals:", radar_velocity_df["is_stable_2d"].sum())

RADAR reconstructable intervals: 97517
RADAR stable intervals: 38179


In [11]:
radar_velocity_df.to_csv(
    RESULT_DIR / "radar_velocity_df.csv",
    index=False,
)

In [12]:
radar_validation_df = build_radar_radial_validation_df(
    radar_radial_df,
    vehicle_reference_df,
)

print(
    "Validated RADAR radial observations:",
    len(radar_validation_df),
)

radar_validation_df.head()

Validated RADAR radial observations: 739477


,annotation_token,sensor,radar_radial_velocity_mps,los_global_x,los_global_y,reference_radial_velocity_mps,radial_residual_mps,radial_abs_error_mps,start_annotation_token,end_annotation_token,instance_token,category
0,fc254d1a106749e48eee764b5cc3b3e0,RADAR_LEFT_BACK,-0.629616,0.980234,-0.197071,0.077747,-0.707363,0.707363,fc254d1a106749e48eee764b5cc3b3e0,6a67433ac8b14eeb929614869ab61b61,0472a737bb8848da835677cb662225f2,vehicle.car
1,e49e954ac91c4de9b240039d3b3b6183,RADAR_LEFT_SIDE,0.021454,-0.116510,-0.990536,-1.150392,1.171846,1.171846,e49e954ac91c4de9b240039d3b3b6183,16a6027e37c5462b99d561859653695e,077999516756420abb114256eb52f214,vehicle.trailer
2,39af30293de545049d4dbf0b499b8341,RADAR_LEFT_SIDE,0.315767,0.070739,-0.996602,-0.301987,0.617753,0.617753,39af30293de545049d4dbf0b499b8341,78d012dab3ea4db285a6718dc0ed5d8b,077999516756420abb114256eb52f214,vehicle.trailer
3,39af30293de545049d4dbf0b499b8341,RADAR_LEFT_SIDE,0.276660,0.082664,-0.995802,-0.299749,0.576409,0.576409,39af30293de545049d4dbf0b499b8341,78d012dab3ea4db285a6718dc0ed5d8b,077999516756420abb114256eb52f214,vehicle.trailer
4,39af30293de545049d4dbf0b499b8341,RADAR_RIGHT_FRONT,-0.117052,0.091763,-0.994866,-0.297939,0.180887,0.180887,39af30293de545049d4dbf0b499b8341,78d012dab3ea4db285a6718dc0ed5d8b,077999516756420abb114256eb52f214,vehicle.trailer


In [13]:
radar_validation_df.to_csv(
    RESULT_DIR / "radar_validation_df.csv",
    index=False,
)

In [15]:
interval_keys = [
    "start_annotation_token",
    "end_annotation_token",
    "instance_token",
    "category",
]

radar_validation_interval_df = (
    radar_validation_df
    .groupby(interval_keys, as_index=False)
    .agg(
        radar_point_count=("radial_abs_error_mps", "size"),
        radial_abs_error_median_mps=("radial_abs_error_mps", "median"),
        radial_signed_bias_mps=("radial_residual_mps", "median"),
    )
)

radar_validation_interval_df.head()

,start_annotation_token,end_annotation_token,instance_token,category,radar_point_count,radial_abs_error_median_mps,radial_signed_bias_mps
0,00006ca85bb74bbab19ae47b86eed8a5,6e7c0caeb5404b4fa9b97ceda591d4ad,5da2689118044deaac1d4e33a9b4edf6,vehicle.car,3,0.173991,-0.173991
1,0000b34c102e4e35b2e186d8683ae68d,b1156e1f84db44788f2a665ce50a8a02,09b511824aaf4ea283415af5bfd9526a,vehicle.trailer,12,0.348497,0.063139
2,000120f7327943fb9b99631d8ee0602c,14163f692766472c86e59c9d64a64b4b,317e68cf5b344627a4d75bd75d5bdcd5,vehicle.car,4,25.870607,25.870607
3,00012f8d63984bbfa71b712cb29200ec,fee6fabd123b41d8a0f2f0a1787ee2b4,e0750dfbe49a414995824ea6ef9351ad,vehicle.car,2,0.195227,-0.195227
4,000279f70e8a4a0db5c5c79072077b49,81aae085cccc43d28bc0a2cc52f8cfdf,c6debf5f3a904b9b93c83b32bba46518,vehicle.truck,2,0.108933,0.108933


In [16]:
errors = radar_validation_interval_df[
    "radial_abs_error_median_mps"
]

radar_native_summary = pd.Series({
    "intervals": len(radar_validation_interval_df),
    "radar_points": len(radar_validation_df),
    "median_abs_radial_error_mps": errors.median(),
    "radial_error_iqr_mps": (
        errors.quantile(0.75) - errors.quantile(0.25)
    ),
    "radial_error_p90_mps": errors.quantile(0.90),
    "median_signed_bias_mps": (
        radar_validation_interval_df[
            "radial_signed_bias_mps"
        ].median()
    ),
})

radar_native_summary

intervals                      118083.000000
radar_points                   739477.000000
median_abs_radial_error_mps         0.220458
radial_error_iqr_mps                0.501528
radial_error_p90_mps               25.434694
median_signed_bias_mps              0.023928
dtype: float64

In [17]:
reference_velocity_df = vehicle_reference_df[
    [
        "start_annotation_token",
        "end_annotation_token",
        "speed_2d_mps",
    ]
]

radar_check_df = radar_velocity_df.merge(
    reference_velocity_df,
    on=["start_annotation_token", "end_annotation_token"],
    how="left",
    validate="one_to_one",
)

radar_check_df["speed_abs_error_mps"] = (
    radar_check_df["radar_speed_2d_mps"]
    - radar_check_df["speed_2d_mps"]
).abs()

stable_df = radar_check_df[
    radar_check_df["is_stable_2d"]
].copy()

In [18]:
radar_2d_summary = pd.Series({
    "stable_intervals": len(stable_df),
    "median_2d_speed_error_mps": (
        stable_df["speed_abs_error_mps"].median()
    ),
    "p90_2d_speed_error_mps": (
        stable_df["speed_abs_error_mps"].quantile(0.90)
    ),
})

radar_2d_summary

stable_intervals             38179.000000
median_2d_speed_error_mps        0.622442
p90_2d_speed_error_mps          41.284618
dtype: float64

In [19]:
comparison_summary = pd.DataFrame({
    "Metric": [
        "Median error (m/s)",
        "P90 error (m/s)",
    ],
    "Native RADAR radial": [
        radar_native_summary["median_abs_radial_error_mps"],
        radar_native_summary["radial_error_p90_mps"],
    ],
    "Reconstructed RADAR 2D": [
        radar_2d_summary["median_2d_speed_error_mps"],
        radar_2d_summary["p90_2d_speed_error_mps"],
    ],
})

comparison_summary

,Metric,Native RADAR radial,Reconstructed RADAR 2D
0,Median error (m/s),0.220458,0.622442
1,P90 error (m/s),25.434694,41.284618


In [20]:
stable_keys = stable_df[
    [
        "start_annotation_token",
        "end_annotation_token",
    ]
].drop_duplicates()

stable_native_df = radar_validation_interval_df.merge(
    stable_keys,
    on=[
        "start_annotation_token",
        "end_annotation_token",
    ],
    how="inner",
)

stable_native_errors = stable_native_df[
    "radial_abs_error_median_mps"
]

stable_native_summary = pd.Series({
    "intervals": len(stable_native_df),
    "median_abs_radial_error_mps": stable_native_errors.median(),
    "radial_error_iqr_mps": (
        stable_native_errors.quantile(0.75)
        - stable_native_errors.quantile(0.25)
    ),
    "radial_error_p90_mps": stable_native_errors.quantile(0.90),
    "median_signed_bias_mps": stable_native_df[
        "radial_signed_bias_mps"
    ].median(),
})

stable_native_summary

intervals                      38179.000000
median_abs_radial_error_mps        0.378900
radial_error_iqr_mps               1.055648
radial_error_p90_mps              48.323413
median_signed_bias_mps             0.066247
dtype: float64

In [21]:
comparison_summary = pd.DataFrame({
    "Metric": [
        "Intervals",
        "Median error (m/s)",
        "P90 error (m/s)",
    ],
    "Native RADAR radial": [
        len(stable_native_df),
        stable_native_summary["median_abs_radial_error_mps"],
        stable_native_summary["radial_error_p90_mps"],
    ],
    "Reconstructed RADAR 2D": [
        len(stable_df),
        radar_2d_summary["median_2d_speed_error_mps"],
        radar_2d_summary["p90_2d_speed_error_mps"],
    ],
})

comparison_summary

,Metric,Native RADAR radial,Reconstructed RADAR 2D
0,Intervals,38179.000000,38179.000000
1,Median error (m/s),0.378900,0.622442
2,P90 error (m/s),48.323413,41.284618
